In [30]:
import carla
import cv2

In [31]:
import numpy as np

In [36]:
CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 10.0
TRAFFIC_MANAGER_PORT = 8000
CAMERA_POS_Z = 3
CAMERA_POS_X = 0


In [15]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT)
   
    world = client.get_world()

    if not world:
        raise RuntimeError(
            "Carla was not able to return a valid world."
        )
    
    print(f'## Client connetion with Carla created with success ##\n')
    return client, world

In [14]:
def load_map(client: carla.Client, world: carla.World, town: str):
    print(town)
    world = client.load_world(town)
    print(f"## World {town} initiallized ##\n")

In [6]:
def configure_traffic_manager(client: carla.Client) -> None :

    traffic_manager = client.get_trafficmanager(TRAFFIC_MANAGER_PORT)
    traffic_manager_port = traffic_manager.get_port()
    print(f"## Traffic maneger port: {traffic_manager_port} ##\n")
    traffic_manager.set_global_distance_to_leading_vehicle(
        1.0
    )


In [ ]:
def spawn_lead_vehicle(world: carla.World) -> carla.Vehicle:

    vehicle_blueprint = world.get_blueprint_library().filter("vehicle.*")

    # for vehicle in vehicle_blueprint:
    #     print(f'Vehicle found: {vehicle.id}\n')

    spawn_points = world.get_map().get_spawn_points()

    vehicle =  world.try_spawn_actor(
        vehicle_blueprint[0],
        spawn_points[10]
    )

    return vehicle


In [8]:
def enable_autopilot(vehicle: carla.Vehicle, autopilot_status: bool):

    if not vehicle:
        raise RuntimeError(
            "No valid vehicle object received."
        )
    
    vehicle.set_autopilot(
        autopilot_status,
        TRAFFIC_MANAGER_PORT,
    )
    print(f"## Autopilot enabled: {autopilot_status} ##\n")

In [32]:
def attach_rgb_camera(world: carla.World, ego_vehicle: carla.Vehicle) -> list[carla.Actor, int, int]:

    camera_bp = world.get_blueprint_library().find('sensor.camera.rgb')
    camera_transform = carla.Transform(carla.Location(z=CAMERA_POS_Z, x=CAMERA_POS_X))
    camera = world.spawn_actor(camera_bp, camera_transform, attach_to=ego_vehicle)

    image_w = camera_bp.get_attribute("image_size_x").as_int()
    image_h = camera_bp.get_attribute("image_size_y").as_int()

    return camera, image_w, image_w


In [33]:
def camera_callback(image, data_dict):
    data_dict['image'] = np.reshape(np.copy(image.raw_data), (image.height, image.width, 4))

In [34]:
def main():

    client, world = connect_to_carla()
    configure_traffic_manager(client)
    #load_map(client, world, "Town02")

    for vehicle in world.get_actors().filter("vehicle.*"):
        vehicle.destroy()
        print(f"Vehicle: {vehicle} destroid.")
        
    lead_vehicle = spawn_lead_vehicle(world)
    camera, image_w, image_h = attach_rgb_camera(world, lead_vehicle)
    camera_data = {'image': np.zeros((image_h, image_w, 4))}
    camera.listen(lambda image: camera_callback(image, camera_data))
    enable_autopilot(lead_vehicle, True)

    cv2.namedWindow('RGB Camera', cv2.WINDOW_AUTOSIZE)
    cv2.imshow('RGB Camera', camera_data['image'])
    cv2.waitKey(1)

    while True:
        cv2.imshow('RGB Camera', camera_data['image'])

        if cv2.waitKey(1) == ord('q'):
            break
    cv2.destroyAllWindows()
    
    for vehicle in world.get_actors().filter("vehicle.*"):
        vehicle.destroy()
        print(f"Vehicle: {vehicle} destroid.")

In [38]:
if __name__ == '__main__':
    main()

## Client connetion with Carla created with success ##

## Traffic maneger port: 8000 ##

## Autopilot enabled: True ##

Vehicle: Actor(id=31, type=vehicle.audi.a2) destroid.
